In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3


In [2]:
def scrape_books():
    print("Scraping books...")
    books_data = []
    page = 1
    
    while len(books_data) < 65:
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"
        response = requests.get(url)
        if response.status_code != 200:
            print(f"Failed to fetch page {page}")
            break
            
        soup = BeautifulSoup(response.content, 'html.parser')
        books = soup.find_all('article', class_='product_pod')
        
        for book in books:
            title = book.h3.a['title']
            price_text = book.find('p', class_='price_color').text
            
            star_classes = book.find('p', class_='star-rating')['class']
            star_rating = [c for c in star_classes if c != 'star-rating'][0]
            
            product_url = "https://books.toscrape.com/catalogue/" + book.h3.a['href']
            prod_response = requests.get(product_url)
            prod_soup = BeautifulSoup(prod_response.content, 'html.parser')
            
            availability = prod_soup.find('p', class_='availability').text.strip()
            
            breadcrumb = prod_soup.find('ul', class_='breadcrumb')
            category = breadcrumb.find_all('li')[2].text.strip()
            
            books_data.append({
                'title': title,
                'price': price_text,
                'star_rating': star_rating,
                'availability': availability,
                'category': category
            })
            
            if len(books_data) >= 65:
                break
                
        page += 1
        
    categories_count = len(set([b['category'] for b in books_data]))
    print(f"Scraped {len(books_data)} books across {categories_count} categories.")
    
    if len(books_data) < 60:
        raise ValueError("Requirement failed: fewer than 60 books scraped.")
    if categories_count < 3:
        raise ValueError("Requirement failed: fewer than 3 categories scraped.")
        
    return pd.DataFrame(books_data)


In [3]:
raw_df = scrape_books()

Scraping books...
Scraped 65 books across 26 categories.


In [4]:
raw_df.head()

,title,price,star_rating,availability,category
0,A Light in the Attic,£51.77,Three,In stock (22 available),Poetry
1,Tipping the Velvet,£53.74,One,In stock (20 available),Historical Fiction
2,Soumission,£50.10,One,In stock (20 available),Fiction
3,Sharp Objects,£47.82,Four,In stock (20 available),Mystery
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock (20 available),History


In [5]:
def clean_and_convert(df):
    print("Cleaning and converting data...")
    df['price_gbp'] = df['price'].str.extract(r'([\d.]+)').astype(float)
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    df['rating'] = pd.to_numeric(df['star_rating'].map(rating_map), errors='coerce')
    df['in_stock'] = df['availability'].str.contains('In stock', case=False, na=False).astype(bool)
    df.dropna(subset=['title', 'price_gbp', 'rating', 'category'], inplace=True)
    df['rating'] = df['rating'].astype(int)
    df['price_inr'] = (df['price_gbp'] * 105.50).round(2)
    return df


In [6]:
clean_df = clean_and_convert(raw_df)

Cleaning and converting data...


In [7]:
display(clean_df.head())
print(clean_df.dtypes)

,title,price,star_rating,availability,category,price_gbp,rating,in_stock,price_inr
0,A Light in the Attic,£51.77,Three,In stock (22 available),Poetry,51.77,3,True,5461.74
1,Tipping the Velvet,£53.74,One,In stock (20 available),Historical Fiction,53.74,1,True,5669.57
2,Soumission,£50.10,One,In stock (20 available),Fiction,50.10,1,True,5285.55
3,Sharp Objects,£47.82,Four,In stock (20 available),Mystery,47.82,4,True,5045.01
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock (20 available),History,54.23,5,True,5721.26


title            object
price            object
star_rating      object
availability     object
category         object
price_gbp       float64
rating            int32
in_stock           bool
price_inr       float64
dtype: object


In [8]:
def load_to_db(df, db_path='books.db'):
    print(f"Loading data into {db_path}...")
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON")
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS categories(
            category_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT UNIQUE
        )
    ''')
    
    cursor.execute('DROP TABLE IF EXISTS books')
    cursor.execute('''
        CREATE TABLE books(
            book_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            price_gbp REAL,
            price_inr REAL,
            rating INTEGER,
            in_stock INTEGER,
            category_id INTEGER,
            FOREIGN KEY(category_id) REFERENCES categories(category_id)
        )
    ''')
    
    unique_categories = df[['category']].drop_duplicates()
    for _, row in unique_categories.iterrows():
        cursor.execute('INSERT OR IGNORE INTO categories (category_name) VALUES (?)', (row['category'],))
        
    category_map = pd.read_sql('SELECT category_id, category_name FROM categories', conn)
    df = df.merge(category_map, left_on='category', right_on='category_name', how='left')
    
    books_to_insert = df[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']]
    books_to_insert.to_sql('books', conn, if_exists='append', index=False)
    conn.commit()
    return conn

conn = load_to_db(clean_df)


Loading data into books.db...


In [9]:
pd.read_sql('SELECT * FROM categories LIMIT 5', conn)

,category_id,category_name
0,1,Poetry
1,2,Historical Fiction
2,3,Fiction
3,4,Mystery
4,5,History


In [10]:
pd.read_sql('SELECT * FROM books LIMIT 5', conn)

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,A Light in the Attic,51.77,5461.74,3,1,1
1,2,Tipping the Velvet,53.74,5669.57,1,1,2
2,3,Soumission,50.10,5285.55,1,1,3
3,4,Sharp Objects,47.82,5045.01,4,1,4
4,5,Sapiens: A Brief History of Humankind,54.23,5721.26,5,1,5


In [11]:
q1 = '''
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4
ORDER BY price_gbp DESC
LIMIT 5;
'''
pd.read_sql(q1, conn)


,title,price_gbp,rating
0,The Past Never Ends,56.50,4
1,Sapiens: A Brief History of Humankind,54.23,5
2,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
3,Behind Closed Doors,52.22,4
4,"We Love You, Charlie Freeman",50.27,5


In [12]:
q2 = '''
SELECT DISTINCT rating
FROM books
ORDER BY rating DESC;
'''
pd.read_sql(q2, conn)


,rating
0,5
1,4
2,3
3,2
4,1


In [13]:
q3 = '''
SELECT title, rating
FROM books
WHERE rating IN (1, 5)
LIMIT 5;
'''
pd.read_sql(q3, conn)


,title,rating
0,Tipping the Velvet,1
1,Soumission,1
2,Sapiens: A Brief History of Humankind,5
3,The Requiem Red,1
4,The Black Maria,1


In [14]:
q4 = '''
SELECT title, price_inr
FROM books
WHERE price_inr BETWEEN 2000 AND 3000
LIMIT 5;
'''
pd.read_sql(q4, conn)


,title,price_inr
0,The Requiem Red,2389.57
1,The Boys in the Boat: Nine Americans and Their...,2384.30
2,Shakespeare's Sonnets,2179.63
3,Olio,2519.34
4,Chase Me (Paris Nights #2),2665.98


In [15]:
q5 = '''
SELECT c.category_name, b.title, b.rating
FROM books b
JOIN categories c ON b.category_id = c.category_id
ORDER BY b.rating DESC, c.category_name ASC
LIMIT 10;
'''
sql_join_df = pd.read_sql(q5, conn)
sql_join_df


,category_name,title,rating
0,Fiction,Private Paris (Private #10),5
1,Fiction,"We Love You, Charlie Freeman",5
2,Fiction,Thirst,5
3,History,Sapiens: A Brief History of Humankind,5
4,Music,Rip it Up and Start Again,5
5,Nonfiction,Worlds Elsewhere: Journeys Around Shakespeare’...,5
6,Nonfiction,#HigherSelfie: Wake Up Your Life. Free Your So...,5
7,Philosophy,Sophie's World,5
8,Romance,Chase Me (Paris Nights #2),5
9,Romance,Black Dust,5


In [16]:
books_df = pd.read_sql("SELECT * FROM books", conn)
categories_df = pd.read_sql("SELECT * FROM categories", conn)

merged_df = pd.merge(books_df, categories_df, on='category_id', how='inner')
merged_df = merged_df[['category_name', 'title', 'rating']]
merged_df = merged_df.sort_values(by=['rating', 'category_name'], ascending=[False, True]).head(10).reset_index(drop=True)

merged_df


,category_name,title,rating
0,Fiction,Private Paris (Private #10),5
1,Fiction,"We Love You, Charlie Freeman",5
2,Fiction,Thirst,5
3,History,Sapiens: A Brief History of Humankind,5
4,Music,Rip it Up and Start Again,5
5,Nonfiction,Worlds Elsewhere: Journeys Around Shakespeare’...,5
6,Nonfiction,#HigherSelfie: Wake Up Your Life. Free Your So...,5
7,Philosophy,Sophie's World,5
8,Romance,Chase Me (Paris Nights #2),5
9,Romance,Black Dust,5


In [17]:
print("Do the outputs match?")
print(sql_join_df.equals(merged_df))


Do the outputs match?
True


In [18]:
conn.close()